In [ ]:
# ================================================================
# CfC / LIQUID NEURAL NETWORK FRAUD DETECTION EXPERIMENT
# ================================================================

# -----------------------------
# 1. INSTALL PACKAGE
# -----------------------------

!pip install ncps -q


# -----------------------------
# 2. IMPORT LIBRARIES
# -----------------------------

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve
)

from ncps.torch import CfC


# Reproducibility
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)


# ================================================================
# 3. LOAD ORIGINAL BANKSIM DATASET
# ================================================================

PATH = "banksim_dataset.csv"

df = pd.read_csv(PATH)

print("\nOriginal shape:", df.shape)
print(df.columns.tolist())


# ================================================================
# 4. REMOVE CONSTANT ZIP COLUMNS
# ================================================================

df = df.drop(
    columns=["zipcodeOri", "zipMerchant"]
)


# ================================================================
# 5. CLEAN QUOTATION MARKS
# ================================================================

text_cols = [
    "customer",
    "age",
    "gender",
    "merchant",
    "category"
]

for col in text_cols:

    df[col] = (
        df[col]
        .astype(str)
        .str.replace("'", "", regex=False)
    )


# ================================================================
# 6. CONVERT AGE
# ================================================================

# Unknown age becomes -1

df["age"] = (
    df["age"]
    .replace("U", "-1")
    .astype(int)
)


# ================================================================
# 7. ENCODE GENDER
# ================================================================

df = pd.get_dummies(
    df,
    columns=["gender"],
    prefix="gender",
    dtype=int
)


# ================================================================
# 8. SORT TRANSACTIONS CHRONOLOGICALLY
# ================================================================

df = (
    df
    .sort_values(["customer", "step"])
    .reset_index(drop=True)
)


# ================================================================
# 9. CREATE TIME GAP
# ================================================================

df["time_delta"] = (
    df.groupby("customer")["step"]
      .diff()
      .fillna(0)
)


# ================================================================
# 10. IMPORTANT:
# REMOVE MERCHANT AND CATEGORY
# ================================================================

df = df.drop(
    columns=[
        "merchant",
        "category"
    ]
)


print("\nColumns after cleaning:")
print(df.columns.tolist())


# ================================================================
# 11. DEFINE MODEL FEATURES
# ================================================================

# Notice:
#
# fraud    IS NOT HERE
# customer IS NOT HERE
# step     IS NOT HERE

feature_cols = [
    "age",
    "amount",
    "time_delta",
    "gender_E",
    "gender_F",
    "gender_M",
    "gender_U"
]


print("\nFEATURES GIVEN TO MODEL:")

for feature in feature_cols:
    print(feature)

print("\nNumber of model features:", len(feature_cols))


# Safety check

assert "fraud" not in feature_cols
assert "customer" not in feature_cols
assert "step" not in feature_cols
assert "merchant" not in feature_cols
assert "category" not in feature_cols

print("\nLeakage safety check PASSED.")


# ================================================================
# 12. CHRONOLOGICAL SPLIT
# ================================================================

train_df = df[
    df["step"] <= 125
].copy()


val_df = df[
    (df["step"] >= 126) &
    (df["step"] <= 152)
].copy()


test_df = df[
    (df["step"] >= 153) &
    (df["step"] <= 179)
].copy()


print("\nTrain:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


# ================================================================
# 13. SCALE NUMERIC FEATURES
# ================================================================

scale_cols = [
    "age",
    "amount",
    "time_delta"
]

scaler = StandardScaler()


# FIT ONLY ON TRAINING DATA

train_df[scale_cols] = scaler.fit_transform(
    train_df[scale_cols]
)


# Validation/test only transformed

val_df[scale_cols] = scaler.transform(
    val_df[scale_cols]
)

test_df[scale_cols] = scaler.transform(
    test_df[scale_cols]
)


# ================================================================
# 14. COMBINE AGAIN FOR SEQUENCE HISTORY
# ================================================================

scaled_df = pd.concat(
    [
        train_df,
        val_df,
        test_df
    ],
    ignore_index=True
)


scaled_df = (
    scaled_df
    .sort_values(["customer", "step"])
    .reset_index(drop=True)
)


# ================================================================
# 15. CREATE TEMPORAL SEQUENCES
# ================================================================

SEQ_LEN = 10


class TemporalFraudDataset(Dataset):

    def __init__(
        self,
        data,
        feature_cols,
        seq_len,
        target_start,
        target_end
    ):

        self.samples = []

        for customer, group in data.groupby(
            "customer",
            sort=False
        ):

            group = (
                group
                .sort_values("step")
                .reset_index(drop=True)
            )

            features = (
                group[feature_cols]
                .values
                .astype(np.float32)
            )

            labels = (
                group["fraud"]
                .values
                .astype(np.float32)
            )

            steps = group["step"].values


            for i in range(
                seq_len - 1,
                len(group)
            ):

                # Current transaction must belong
                # to required time period

                if (
                    target_start
                    <= steps[i]
                    <= target_end
                ):

                    start = i - seq_len + 1

                    x = features[
                        start:i + 1
                    ]

                    y = labels[i]

                    self.samples.append(
                        (x, y)
                    )


    def __len__(self):

        return len(self.samples)


    def __getitem__(self, idx):

        x, y = self.samples[idx]

        return (
            torch.tensor(
                x,
                dtype=torch.float32
            ),

            torch.tensor(
                y,
                dtype=torch.float32
            )
        )


# ================================================================
# 16. CREATE DATASETS
# ================================================================

train_dataset = TemporalFraudDataset(
    scaled_df,
    feature_cols,
    SEQ_LEN,
    0,
    125
)


val_dataset = TemporalFraudDataset(
    scaled_df,
    feature_cols,
    SEQ_LEN,
    126,
    152
)


test_dataset = TemporalFraudDataset(
    scaled_df,
    feature_cols,
    SEQ_LEN,
    153,
    179
)


print("\nSequences:")

print(
    "Train:",
    len(train_dataset)
)

print(
    "Validation:",
    len(val_dataset)
)

print(
    "Test:",
    len(test_dataset)
)


# ================================================================
# 17. DATA LOADERS
# ================================================================

BATCH_SIZE = 256


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Check input

x_test, y_test = next(
    iter(train_loader)
)

print(
    "\nInput batch shape:",
    x_test.shape
)

print(
    "Label shape:",
    y_test.shape
)


# ================================================================
# 18. CALCULATE CLASS WEIGHT FROM TRAINING SEQUENCES
# ================================================================

train_labels = np.array([
    y.item()
    for _, y in train_dataset
])


n_fraud = np.sum(
    train_labels == 1
)

n_normal = np.sum(
    train_labels == 0
)


pos_weight_value = (
    n_normal / n_fraud
)


pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32
).to(DEVICE)


print("\nTraining normal:", n_normal)
print("Training fraud:", n_fraud)

print(
    "Fraud class weight:",
    pos_weight.item()
)


# ================================================================
# 19. BUILD CfC LIQUID NETWORK
# ================================================================

class LiquidFraudDetector(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size=64
    ):

        super().__init__()


        self.cfc = CfC(
            input_size,
            hidden_size,
            batch_first=True
        )


        self.classifier = nn.Sequential(

            nn.Linear(
                hidden_size,
                16
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                16,
                1
            )
        )


    def forward(self, x):

        output, _ = self.cfc(x)

        last_output = output[:, -1, :]

        logit = self.classifier(
            last_output
        )

        return logit.squeeze(-1)


model = LiquidFraudDetector(
    input_size=len(feature_cols),
    hidden_size=64
).to(DEVICE)


print("\nMODEL:")
print(model)


# ================================================================
# 20. LOSS + OPTIMIZER
# ================================================================

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# ================================================================
# 21. TRAINING FUNCTION
# ================================================================

def train_epoch():

    model.train()

    total_loss = 0


    for x, y in train_loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)


        optimizer.zero_grad()


        logits = model(x)


        loss = criterion(
            logits,
            y
        )


        loss.backward()

        optimizer.step()


        total_loss += (
            loss.item()
            * x.size(0)
        )


    return (
        total_loss
        / len(train_dataset)
    )


# ================================================================
# 22. GET PROBABILITIES
# ================================================================

def get_predictions(loader):

    model.eval()

    probabilities = []
    labels = []


    with torch.no_grad():

        for x, y in loader:

            x = x.to(DEVICE)

            logits = model(x)

            probs = torch.sigmoid(
                logits
            )


            probabilities.extend(
                probs.cpu().numpy()
            )

            labels.extend(
                y.numpy()
            )


    return (
        np.array(probabilities),
        np.array(labels)
    )


# ================================================================
# 23. TRAIN MODEL
# ================================================================

EPOCHS = 15


print("\nTRAINING STARTED\n")


for epoch in range(
    1,
    EPOCHS + 1
):

    loss = train_epoch()


    val_probs, val_labels = (
        get_predictions(
            val_loader
        )
    )


    pr_auc = (
        average_precision_score(
            val_labels,
            val_probs
        )
    )


    roc_auc = (
        roc_auc_score(
            val_labels,
            val_probs
        )
    )


    print(
        f"Epoch {epoch}/{EPOCHS}"
        f" | Loss: {loss:.4f}"
        f" | Val PR-AUC: {pr_auc:.4f}"
        f" | Val ROC-AUC: {roc_auc:.4f}"
    )


# ================================================================
# 24. FIND THRESHOLD USING VALIDATION ONLY
# ================================================================

val_probs, val_labels = (
    get_predictions(
        val_loader
    )
)


precision_arr, recall_arr, thresholds = (
    precision_recall_curve(
        val_labels,
        val_probs
    )
)


f1_scores = (
    2
    * precision_arr[:-1]
    * recall_arr[:-1]
    /
    (
        precision_arr[:-1]
        + recall_arr[:-1]
        + 1e-10
    )
)


best_index = np.argmax(
    f1_scores
)


BEST_THRESHOLD = (
    thresholds[best_index]
)


print(
    "\nBest validation threshold:",
    round(
        float(BEST_THRESHOLD),
        4
    )
)

print(
    "Validation F1:",
    round(
        float(
            f1_scores[best_index]
        ),
        4
    )
)


# ================================================================
# 25. FINAL TEST
# ================================================================

test_probs, test_labels = (
    get_predictions(
        test_loader
    )
)


test_predictions = (
    test_probs
    >= BEST_THRESHOLD
).astype(int)


accuracy = accuracy_score(
    test_labels,
    test_predictions
)


precision = precision_score(
    test_labels,
    test_predictions,
    zero_division=0
)


recall = recall_score(
    test_labels,
    test_predictions,
    zero_division=0
)


f1 = f1_score(
    test_labels,
    test_predictions,
    zero_division=0
)


roc_auc = roc_auc_score(
    test_labels,
    test_probs
)


pr_auc = average_precision_score(
    test_labels,
    test_probs
)


# ================================================================
# 26. RESULTS
# ================================================================

print("\n")
print("=" * 50)

print(
    "FINAL CLEAN CfC TEST RESULTS"
)

print("=" * 50)


print(
    "Threshold :",
    round(
        float(BEST_THRESHOLD),
        4
    )
)

print(
    "Accuracy  :",
    round(
        accuracy,
        4
    )
)

print(
    "Precision :",
    round(
        precision,
        4
    )
)

print(
    "Recall    :",
    round(
        recall,
        4
    )
)

print(
    "F1 Score  :",
    round(
        f1,
        4
    )
)

print(
    "ROC-AUC   :",
    round(
        roc_auc,
        4
    )
)

print(
    "PR-AUC    :",
    round(
        pr_auc,
        4
    )
)


print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(
        test_labels,
        test_predictions
    )
)


print(
    "\nClassification Report:"
)

print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=[
            "Normal",
            "Fraud"
        ],
        digits=4
    )
)